# 03 — Functional Connectivity Analysis

This notebook converts the motion-censored 216-ROI resting-state
time series into functional connectivity matrices for Rest 1 and Rest 2.

For each session, censored time points are removed and pairwise Pearson
correlations are calculated across the 216 brain regions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROI_DIR = Path(
    r"C:\Users\rahin\outputs\roi_timeseries_216"
)

roi_files = sorted(ROI_DIR.glob("*_216ROI.csv"))

print("ROI files found:", len(roi_files))
print("Expected:", 36)

## 1. Single-Session Functional Connectivity Test

A single session is tested first to verify that the censored ROI time series
can be converted into a 216 × 216 Pearson-correlation matrix.

In [ ]:
TEST_FILE = ROI_DIR / "E3746_Rest1_216ROI.csv"

test_df = pd.read_csv(TEST_FILE)

print("Original shape:", test_df.shape)

# Remove censored TRs
test_clean = test_df.dropna(axis=0, how="all")

print("Usable time-series shape:", test_clean.shape)

# Functional connectivity matrix
test_fc = test_clean.corr(method="pearson")

print("Connectivity matrix shape:", test_fc.shape)

print(
    "Contains NaN values:",
    test_fc.isna().any().any()
)

print(
    "Symmetric:",
    np.allclose(
        test_fc.values,
        test_fc.values.T,
        equal_nan=True
    )
)

## 2. Batch Functional Connectivity Matrices

Pearson functional connectivity matrices are calculated for all Rest 1
and Rest 2 sessions after removing motion-censored time points.

In [ ]:
FC_DIR = Path("outputs") / "functional_connectivity_216"
FC_DIR.mkdir(parents=True, exist_ok=True)

fc_summary = []

for roi_file in roi_files:

    # Load ROI time series
    df = pd.read_csv(roi_file)

    # Remove motion-censored TRs
    clean_df = df.dropna(axis=0, how="all")

    # Safety check
    if clean_df.shape[1] != 216:
        raise ValueError(
            f"{roi_file.name}: expected 216 ROIs, "
            f"found {clean_df.shape[1]}"
        )

    # Pearson functional connectivity
    fc = clean_df.corr(method="pearson")

    # Check matrix
    if fc.shape != (216, 216):
        raise ValueError(
            f"{roi_file.name}: unexpected FC shape {fc.shape}"
        )

    if fc.isna().any().any():
        raise ValueError(
            f"{roi_file.name}: FC contains NaN values"
        )

    if not np.allclose(fc.values, fc.values.T):
        raise ValueError(
            f"{roi_file.name}: FC matrix is not symmetric"
        )

    # Output filename
    output_name = roi_file.name.replace(
        "_216ROI.csv",
        "_FC_216.csv"
    )

    output_file = FC_DIR / output_name

    fc.to_csv(output_file)

    fc_summary.append({
        "Input": roi_file.name,
        "Usable_TRs": clean_df.shape[0],
        "FC_Rows": fc.shape[0],
        "FC_Columns": fc.shape[1],
        "Symmetric": True
    })

    print(
        f"{roi_file.stem} | "
        f"TRs={clean_df.shape[0]} | "
        f"FC={fc.shape} | OK"
    )


fc_summary_df = pd.DataFrame(fc_summary)

print("\nFUNCTIONAL CONNECTIVITY COMPLETE")
print("Matrices created:", len(fc_summary_df))

## 3. Pre-to-Post Functional Connectivity Change

For each participant, neurofeedback-related functional network
reorganization is represented as the difference between the post-
and pre-neurofeedback connectivity matrices:

\[
\Delta FC = FC_{Rest2} - FC_{Rest1}
\]

Positive values indicate increased functional connectivity from Rest 1
to Rest 2, whereas negative values indicate decreased connectivity.

In [ ]:
# Test pre-to-post connectivity change on E3746

rest1_fc_file = FC_DIR / "E3746_Rest1_FC_216.csv"
rest2_fc_file = FC_DIR / "E3746_Rest2_FC_216.csv"

rest1_fc = pd.read_csv(
    rest1_fc_file,
    index_col=0
)

rest2_fc = pd.read_csv(
    rest2_fc_file,
    index_col=0
)

print("Rest1 FC shape:", rest1_fc.shape)
print("Rest2 FC shape:", rest2_fc.shape)

# Calculate connectivity change
delta_fc = rest2_fc - rest1_fc

print("Delta FC shape:", delta_fc.shape)

print(
    "Contains NaN:",
    delta_fc.isna().any().any()
)

print(
    "Symmetric:",
    np.allclose(
        delta_fc.values,
        delta_fc.values.T
    )
)

print(
    "Mean absolute connectivity change:",
    np.mean(np.abs(delta_fc.values))
)

### 3.1 Batch Pre-to-Post Connectivity Change

Pre-to-post functional connectivity change is calculated separately for
each participant. The mean absolute connectivity change provides a
subject-level summary of the magnitude of whole-brain network
reorganization.

In [ ]:
DELTA_DIR = Path("outputs") / "delta_connectivity_216"
DELTA_DIR.mkdir(parents=True, exist_ok=True)

delta_summary = []

for subject in [
    "E3746","E3799","E3973","E4051","E4209","E4253",
    "E4324","E4350","E4360","E4484","E4689","E4697",
    "E4745","E5215","E5580","E5586","E5693","E5694"
]:

    rest1_file = FC_DIR / f"{subject}_Rest1_FC_216.csv"
    rest2_file = FC_DIR / f"{subject}_Rest2_FC_216.csv"

    rest1_fc = pd.read_csv(rest1_file, index_col=0)
    rest2_fc = pd.read_csv(rest2_file, index_col=0)

    # Pre-to-post change
    delta_fc = rest2_fc - rest1_fc

    # Save individual ΔFC matrix
    delta_file = DELTA_DIR / f"{subject}_DeltaFC_216.csv"
    delta_fc.to_csv(delta_file)

    # Exclude diagonal from summary metric
    delta_values = delta_fc.values.copy()

    np.fill_diagonal(
        delta_values,
        np.nan
    )

    mean_abs_delta = np.nanmean(
        np.abs(delta_values)
    )

    mean_signed_delta = np.nanmean(
        delta_values
    )

    delta_summary.append({
        "Subject": subject,
        "Mean_Absolute_DeltaFC": mean_abs_delta,
        "Mean_Signed_DeltaFC": mean_signed_delta
    })

    print(
        f"{subject} | "
        f"Mean |DeltaFC| = {mean_abs_delta:.4f} | "
        f"Mean DeltaFC = {mean_signed_delta:.4f}"
    )


delta_summary_df = pd.DataFrame(delta_summary)

print("\nDELTA CONNECTIVITY COMPLETE")
print("Participants:", len(delta_summary_df))

delta_summary_df

## 4. Neurofeedback Group Comparison

Participant-level functional connectivity changes are linked with the
experimental treatment assignment to compare network reorganization
between the active neurofeedback and sham-control groups.

In [ ]:
PARTICIPANTS_FILE = Path(
    r"C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\participants.tsv"
)

participants = pd.read_csv(
    PARTICIPANTS_FILE,
    sep="\t"
)

# Keep relevant columns
group_info = participants[
    ["Exam", "Group"]
].copy()

# Normalize Exam IDs
group_info["Exam"] = (
    group_info["Exam"]
    .astype(str)
    .str.strip()
)

# One row per Exam ID
group_info = group_info.drop_duplicates(
    subset="Exam"
)

# Merge treatment group with DeltaFC results
delta_group_df = delta_summary_df.merge(
    group_info,
    left_on="Subject",
    right_on="Exam",
    how="left"
)

delta_group_df = delta_group_df[
    [
        "Subject",
        "Group",
        "Mean_Absolute_DeltaFC",
        "Mean_Signed_DeltaFC"
    ]
]

print("Participants:", len(delta_group_df))

print("\nMissing group labels:")
print(delta_group_df["Group"].isna().sum())

print("\nGroup counts:")
print(delta_group_df["Group"].value_counts())

print("\nGroup means:")
print(
    delta_group_df.groupby("Group")[
        ["Mean_Absolute_DeltaFC",
         "Mean_Signed_DeltaFC"]
    ].mean()
)

delta_group_df

### 4.1 Statistical Comparison of Network Reorganization

Differences in participant-level network reorganization between the
active and sham groups are evaluated using Welch's independent-samples
t-test, which does not assume equal group variances. Effect size is
also calculated to quantify the magnitude of the group difference.

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

active = delta_group_df[
    delta_group_df["Group"] == "active"
]["Mean_Absolute_DeltaFC"].values

sham = delta_group_df[
    delta_group_df["Group"] == "sham"
]["Mean_Absolute_DeltaFC"].values

# Welch's independent-samples t-test
t_stat, p_value = ttest_ind(
    active,
    sham,
    equal_var=False
)

# Descriptive statistics
active_mean = np.mean(active)
sham_mean = np.mean(sham)

active_sd = np.std(active, ddof=1)
sham_sd = np.std(sham, ddof=1)

# Pooled SD for Cohen's d
n1 = len(active)
n2 = len(sham)

pooled_sd = np.sqrt(
    (
        (n1 - 1) * active_sd**2 +
        (n2 - 1) * sham_sd**2
    )
    /
    (n1 + n2 - 2)
)

cohens_d = (
    active_mean - sham_mean
) / pooled_sd

print("ACTIVE")
print("n =", n1)
print("Mean =", round(active_mean, 4))
print("SD =", round(active_sd, 4))

print("\nSHAM")
print("n =", n2)
print("Mean =", round(sham_mean, 4))
print("SD =", round(sham_sd, 4))

print("\nGROUP COMPARISON")
print("Mean difference =", round(active_mean - sham_mean, 4))
print("Welch t =", round(t_stat, 4))
print("p =", round(p_value, 4))
print("Cohen's d =", round(cohens_d, 4))

### 4.2 Visualization of Individual Network Reorganization

Individual values are visualized to show the distribution and
between-subject variability in whole-brain functional connectivity
reorganization within the active and sham groups.

In [ ]:
import matplotlib.pyplot as plt

active_values = delta_group_df[
    delta_group_df["Group"] == "active"
]["Mean_Absolute_DeltaFC"]

sham_values = delta_group_df[
    delta_group_df["Group"] == "sham"
]["Mean_Absolute_DeltaFC"]

plt.figure(figsize=(6, 5))

plt.boxplot(
    [active_values, sham_values],
    tick_labels=["Active", "Sham"]
)

# Add individual participant values
for i, values in enumerate(
    [active_values, sham_values],
    start=1
):
    plt.scatter(
        [i] * len(values),
        values,
        alpha=0.7
    )

plt.ylabel("Mean Absolute ΔFC")
plt.xlabel("Neurofeedback Group")

plt.title(
    "Whole-Brain Functional Connectivity Reorganization"
)

plt.tight_layout()
plt.show()

In [ ]:
%pip install matplotlib

## 5 — Fisher-Z Connectivity Matrices

For connection-level group analysis, Pearson correlation coefficients
are transformed using the Fisher r-to-z transformation. This improves
the statistical properties of correlation values for between-subject
comparisons.

The diagonal is excluded because self-correlations are equal to 1.

In [ ]:
ZFC_DIR = Path("outputs") / "functional_connectivity_216_fisherz"
ZFC_DIR.mkdir(parents=True, exist_ok=True)

z_files_created = 0

for fc_file in sorted(FC_DIR.glob("*_FC_216.csv")):

    fc = pd.read_csv(
        fc_file,
        index_col=0
    )

    r = fc.to_numpy(dtype=float)

    # Prevent infinite values at exactly +1 or -1
    r = np.clip(
        r,
        -0.999999,
        0.999999
    )

    # Fisher r-to-z transformation
    z = np.arctanh(r)

    # Self-connections are not analyzed
    np.fill_diagonal(z, np.nan)

    z_df = pd.DataFrame(
        z,
        index=fc.index,
        columns=fc.columns
    )

    output_name = fc_file.name.replace(
        "_FC_216.csv",
        "_FC_Z_216.csv"
    )

    z_df.to_csv(
        ZFC_DIR / output_name
    )

    z_files_created += 1


print("Fisher-Z matrices created:", z_files_created)
print("Expected:", 36)
print("Output directory:", ZFC_DIR)

### 5.1 Fisher-Z Connectivity Change

For each participant, connectivity reorganization is calculated as the
difference between post-neurofeedback (Rest2) and pre-neurofeedback
(Rest1) Fisher-z connectivity matrices.

Group-average change matrices are then calculated separately for the
active and sham neurofeedback groups.

In [ ]:
ZDELTA_DIR = Path("outputs") / "delta_connectivity_216_fisherz"
ZDELTA_DIR.mkdir(parents=True, exist_ok=True)

active_delta = []
sham_delta = []

for _, row in delta_group_df.iterrows():

    subject = row["Subject"]
    group = row["Group"]

    rest1_file = ZFC_DIR / f"{subject}_Rest1_FC_Z_216.csv"
    rest2_file = ZFC_DIR / f"{subject}_Rest2_FC_Z_216.csv"

    rest1 = pd.read_csv(rest1_file, index_col=0)
    rest2 = pd.read_csv(rest2_file, index_col=0)

    delta = rest2 - rest1

    # Save participant-level Fisher-Z change matrix
    delta.to_csv(
        ZDELTA_DIR / f"{subject}_DeltaFC_Z_216.csv"
    )

    if group == "active":
        active_delta.append(delta.to_numpy())

    elif group == "sham":
        sham_delta.append(delta.to_numpy())


active_delta = np.stack(active_delta)
sham_delta = np.stack(sham_delta)

active_mean_delta = np.nanmean(active_delta, axis=0)
sham_mean_delta = np.nanmean(sham_delta, axis=0)

group_difference = (
    active_mean_delta - sham_mean_delta
)

print("Active participants:", active_delta.shape[0])
print("Sham participants:", sham_delta.shape[0])

print("\nMatrix dimensions:")
print("Active:", active_mean_delta.shape)
print("Sham:", sham_mean_delta.shape)
print("Active - Sham:", group_difference.shape)

print("\nNaN values off diagonal:",
      np.isnan(group_difference[~np.eye(216, dtype=bool)]).sum())

print("Symmetric:",
      np.allclose(
          group_difference,
          group_difference.T,
          equal_nan=True
      ))

### 5.2 Active-versus-Sham Connectivity Reorganization Map

The group-difference matrix represents the difference in mean
Fisher-z connectivity change between active and sham neurofeedback:

\[
\Delta Z_{\mathrm{Active}}-\Delta Z_{\mathrm{Sham}}
\]

Positive values indicate connections showing relatively greater
increases in connectivity in the active group, whereas negative values
indicate relatively greater increases in the sham group or decreases
in the active group.

In [ ]:
plt.figure(figsize=(8, 7))

plt.imshow(
    group_difference,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(
    label="Active − Sham Δ Fisher-Z Connectivity"
)

plt.xlabel("ROI")
plt.ylabel("ROI")

plt.title(
    "Active vs Sham Difference in Functional Connectivity Reorganization"
)

plt.tight_layout()
plt.show()

## 6 — Network-Level Connectivity Reorganization

The 216 ROI connectome is summarized into larger functional systems
to determine whether neurofeedback-related reorganization is
concentrated within or between specific brain networks.

In [ ]:
from nilearn import datasets

schaefer_info = datasets.fetch_atlas_schaefer_2018(
    n_rois=200,
    yeo_networks=7,
    resolution_mm=2
)

print("Number of labels:", len(schaefer_info.labels))

print("\nFirst 20 labels:")
for i, label in enumerate(schaefer_info.labels[:20], start=1):
    print(i, label)

In [ ]:
# Remove the background label
schaefer_labels = schaefer_info.labels[1:]

print("Cortical ROI labels:", len(schaefer_labels))

# Convert bytes to strings if necessary
schaefer_labels = [
    label.decode("utf-8") if isinstance(label, bytes) else str(label)
    for label in schaefer_labels
]

# Extract the Yeo-7 network name
schaefer_networks = []

for label in schaefer_labels:
    parts = label.split("_")
    network = parts[2]
    schaefer_networks.append(network)

# Add the 16 Tian subcortical ROIs as one Subcortical system
roi_networks = schaefer_networks + ["Subcortical"] * 16

print("Total ROI assignments:", len(roi_networks))

print("\nNetwork counts:")
print(pd.Series(roi_networks).value_counts())

### 6.1 Network-Level Active-versus-Sham Reorganization

The ROI-level active-versus-sham difference matrix is summarized
according to the seven Schaefer cortical networks and one combined
subcortical system. Each matrix cell represents the mean Fisher-z
connectivity-change difference across all ROI pairs belonging to the
corresponding pair of systems.

In [ ]:
network_order = [
    "Vis",
    "SomMot",
    "DorsAttn",
    "SalVentAttn",
    "Limbic",
    "Cont",
    "Default",
    "Subcortical"
]

roi_networks_array = np.array(roi_networks)

network_difference = pd.DataFrame(
    index=network_order,
    columns=network_order,
    dtype=float
)

for net1 in network_order:
    for net2 in network_order:

        idx1 = np.where(
            roi_networks_array == net1
        )[0]

        idx2 = np.where(
            roi_networks_array == net2
        )[0]

        block = group_difference[
            np.ix_(idx1, idx2)
        ]

        # For within-network blocks, avoid counting
        # diagonal self-connections.
        if net1 == net2:
            block = block.copy()

            if block.shape[0] == block.shape[1]:
                np.fill_diagonal(block, np.nan)

        network_difference.loc[
            net1, net2
        ] = np.nanmean(block)


print("Network matrix shape:",
      network_difference.shape)

print("\nActive - Sham mean ΔFisher-Z:")
display(network_difference.round(4))

### 6.2 Participant-Level Dorsal Attention–Salience/Ventral Attention Change

The dorsal attention–salience/ventral attention connection showed the
largest descriptive active-versus-sham difference. Participant-level
network-pair changes are therefore extracted to examine the
distribution of this effect across individuals.

In [ ]:
dors_idx = np.where(
    roi_networks_array == "DorsAttn"
)[0]

sal_idx = np.where(
    roi_networks_array == "SalVentAttn"
)[0]

network_subject_results = []

for _, row in delta_group_df.iterrows():

    subject = row["Subject"]
    group = row["Group"]

    delta_file = (
        ZDELTA_DIR /
        f"{subject}_DeltaFC_Z_216.csv"
    )

    delta = pd.read_csv(
        delta_file,
        index_col=0
    ).to_numpy()

    block = delta[
        np.ix_(dors_idx, sal_idx)
    ]

    mean_change = np.nanmean(block)

    network_subject_results.append({
        "Subject": subject,
        "Group": group,
        "DorsAttn_SalVentAttn_DeltaZ": mean_change
    })


network_subject_df = pd.DataFrame(
    network_subject_results
)

print(
    network_subject_df.groupby("Group")[
        "DorsAttn_SalVentAttn_DeltaZ"
    ].agg(["count", "mean", "std"])
)

display(network_subject_df)

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

active_net = network_subject_df[
    network_subject_df["Group"] == "active"
]["DorsAttn_SalVentAttn_DeltaZ"].values

sham_net = network_subject_df[
    network_subject_df["Group"] == "sham"
]["DorsAttn_SalVentAttn_DeltaZ"].values

# Welch's t-test
t_stat, p_value = ttest_ind(
    active_net,
    sham_net,
    equal_var=False
)

# Cohen's d
n1 = len(active_net)
n2 = len(sham_net)

sd1 = np.std(active_net, ddof=1)
sd2 = np.std(sham_net, ddof=1)

pooled_sd = np.sqrt(
    ((n1 - 1) * sd1**2 +
     (n2 - 1) * sd2**2)
    /
    (n1 + n2 - 2)
)

cohens_d = (
    np.mean(active_net) -
    np.mean(sham_net)
) / pooled_sd

print("Active mean:", round(np.mean(active_net), 4))
print("Sham mean:", round(np.mean(sham_net), 4))
print("Mean difference:", round(
    np.mean(active_net) - np.mean(sham_net), 4
))

print("\nWelch t:", round(t_stat, 4))
print("p:", round(p_value, 4))
print("Cohen's d:", round(cohens_d, 4))

### 6.3 Multiple-Comparison-Aware Network Analysis

To avoid selecting network pairs based only on the largest observed
group difference, all unique within- and between-network connectivity
changes are evaluated systematically. Statistical significance is
subsequently assessed after correction for multiple comparisons.

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from itertools import combinations_with_replacement

network_test_results = []

# Test all 36 unique within/between-network pairs
for net1, net2 in combinations_with_replacement(network_order, 2):

    idx1 = np.where(roi_networks_array == net1)[0]
    idx2 = np.where(roi_networks_array == net2)[0]

    active_values = []
    sham_values = []

    # Active participants
    for delta in active_delta:

        block = delta[np.ix_(idx1, idx2)].copy()

        # Remove self-connections for within-network comparisons
        if net1 == net2:
            np.fill_diagonal(block, np.nan)

        active_values.append(
            np.nanmean(block)
        )

    # Sham participants
    for delta in sham_delta:

        block = delta[np.ix_(idx1, idx2)].copy()

        if net1 == net2:
            np.fill_diagonal(block, np.nan)

        sham_values.append(
            np.nanmean(block)
        )

    active_values = np.array(active_values)
    sham_values = np.array(sham_values)

    # Welch t-test
    t_stat, p_value = ttest_ind(
        active_values,
        sham_values,
        equal_var=False
    )

    # Cohen's d
    n1 = len(active_values)
    n2 = len(sham_values)

    sd1 = np.std(active_values, ddof=1)
    sd2 = np.std(sham_values, ddof=1)

    pooled_sd = np.sqrt(
        (
            (n1 - 1) * sd1**2 +
            (n2 - 1) * sd2**2
        )
        /
        (n1 + n2 - 2)
    )

    d = (
        np.mean(active_values) -
        np.mean(sham_values)
    ) / pooled_sd

    network_test_results.append({
        "Network_1": net1,
        "Network_2": net2,
        "Active_Mean": np.mean(active_values),
        "Sham_Mean": np.mean(sham_values),
        "Difference": (
            np.mean(active_values) -
            np.mean(sham_values)
        ),
        "t": t_stat,
        "p_uncorrected": p_value,
        "Cohens_d": d
    })


network_stats_df = pd.DataFrame(
    network_test_results
)

# Benjamini-Hochberg FDR correction across 36 tests
reject, p_fdr, _, _ = multipletests(
    network_stats_df["p_uncorrected"],
    alpha=0.05,
    method="fdr_bh"
)

network_stats_df["p_FDR"] = p_fdr
network_stats_df["FDR_significant"] = reject

# Sort by uncorrected p-value
network_stats_df = network_stats_df.sort_values(
    "p_uncorrected"
).reset_index(drop=True)

print("Network comparisons:", len(network_stats_df))

print(
    "Uncorrected p < .05:",
    (network_stats_df["p_uncorrected"] < 0.05).sum()
)

print(
    "FDR-significant:",
    network_stats_df["FDR_significant"].sum()
)

display(
    network_stats_df.head(10).round(4)
)

In [ ]:
%pip install statsmodels

In [ ]:
print("Available participant/group-related variables:\n")

for name in sorted(globals()):
    if any(word in name.lower() for word in
           ["subject", "exam", "group", "network", "delta"]):
        obj = globals()[name]
        try:
            shape = obj.shape
        except:
            shape = None
        print(f"{name:35s} {type(obj).__name__:20s} {shape}")

In [ ]:
print("delta_summary_df columns:")
print(delta_summary_df.columns.tolist())

print("\ndelta_summary_df:")
print(delta_summary_df.to_string(index=False))

print("\nnetwork_subject_df columns:")
print(network_subject_df.columns.tolist())

print("\nFirst rows of network_subject_df:")
print(network_subject_df.head(10).to_string(index=False))

In [ ]:
from pathlib import Path

print("FC-related directories:")
for name in sorted(globals()):
    if "dir" in name.lower() or "fc" in name.lower():
        obj = globals()[name]
        if isinstance(obj, (Path, str)):
            print(f"{name:25s} -> {obj}")

print("\nE4745-related files:")
project_root = Path.cwd()

for f in project_root.rglob("*E4745*"):
    if f.is_file():
        print(f)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------------------------------------
# Representative participant:
# E4745 has mean absolute ΔFC = 0.1347,
# close to the cohort median.
# -------------------------------------------------------
subject_fig = "E4745"

rest1_path = ZFC_DIR / f"{subject_fig}_Rest1_FC_Z_216.csv"
rest2_path = ZFC_DIR / f"{subject_fig}_Rest2_FC_Z_216.csv"

rest1_z = pd.read_csv(rest1_path, index_col=0).to_numpy()
rest2_z = pd.read_csv(rest2_path, index_col=0).to_numpy()

print("Rest1 shape:", rest1_z.shape)
print("Rest2 shape:", rest2_z.shape)

# -------------------------------------------------------
# Aggregate 216 ROI-level FC values into the same
# 8 large-scale functional systems used previously.
# -------------------------------------------------------
def aggregate_to_networks(fc_matrix, roi_networks, network_order):

    network_matrix = np.zeros(
        (len(network_order), len(network_order))
    )

    for i, net_i in enumerate(network_order):
        idx_i = np.where(roi_networks == net_i)[0]

        for j, net_j in enumerate(network_order):
            idx_j = np.where(roi_networks == net_j)[0]

            block = fc_matrix[np.ix_(idx_i, idx_j)].copy()

            # For within-network connectivity,
            # exclude self-connections on the diagonal.
            if i == j:
                mask = ~np.eye(
                    len(idx_i),
                    dtype=bool
                )
                values = block[mask]
            else:
                values = block.ravel()

            values = values[np.isfinite(values)]

            network_matrix[i, j] = np.mean(values)

    return network_matrix


rest1_network = aggregate_to_networks(
    rest1_z,
    roi_networks_array,
    network_order
)

rest2_network = aggregate_to_networks(
    rest2_z,
    roi_networks_array,
    network_order
)

delta_network = rest2_network - rest1_network

print("\nNetwork matrices:", rest1_network.shape)
print("Mean absolute network ΔZ:",
      np.mean(np.abs(delta_network[np.triu_indices(8)])))

# -------------------------------------------------------
# Plot
# -------------------------------------------------------
fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5),
    constrained_layout=True
)

# Use the same scale for Rest1 and Rest2
fc_limit = np.nanmax(
    np.abs(
        np.concatenate([
            rest1_network.ravel(),
            rest2_network.ravel()
        ])
    )
)

# Separate symmetric scale for change
delta_limit = np.nanmax(np.abs(delta_network))

images = []

images.append(
    axes[0].imshow(
        rest1_network,
        vmin=-fc_limit,
        vmax=fc_limit
    )
)

images.append(
    axes[1].imshow(
        rest2_network,
        vmin=-fc_limit,
        vmax=fc_limit
    )
)

images.append(
    axes[2].imshow(
        delta_network,
        vmin=-delta_limit,
        vmax=delta_limit
    )
)

titles = [
    "(A) Baseline Rest1",
    "(B) Post-intervention Rest2",
    "(C) Network transition: Rest2 − Rest1"
]

for ax, title in zip(axes, titles):

    ax.set_title(title, fontsize=12)

    ax.set_xticks(range(len(network_order)))
    ax.set_yticks(range(len(network_order)))

    ax.set_xticklabels(
        network_order,
        rotation=45,
        ha="right",
        fontsize=8
    )

    ax.set_yticklabels(
        network_order,
        fontsize=8
    )

# Shared colorbar for Rest1 / Rest2
cbar1 = fig.colorbar(
    images[1],
    ax=axes[:2],
    shrink=0.80
)
cbar1.set_label("Fisher-z connectivity")

# Separate colorbar for Δ
cbar2 = fig.colorbar(
    images[2],
    ax=axes[2],
    shrink=0.80
)
cbar2.set_label("Δ Fisher-z connectivity") 

plt.savefig(
    "Fig3_Individual_Network_Transition.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("Group-level variables:\n")

print("delta_group_df:")
print(delta_group_df.to_string(index=False))

print("\nnetwork_difference:")
print(network_difference.round(4).to_string())

print("\nnetwork_stats_df columns:")
print(network_stats_df.columns.tolist())

print("\nTop network tests:")
print(
    network_stats_df
    .sort_values("p_value")
    .head(10)
    .to_string(index=False)
)

In [ ]:
print(
    network_stats_df
    .sort_values("p_uncorrected")
    .head(10)
    .to_string(index=False)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Figure 4A: Active - Sham difference heatmap
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))

limit = np.nanmax(np.abs(network_difference.to_numpy()))

im = ax.imshow(
    network_difference.to_numpy(),
    vmin=-limit,
    vmax=limit
)

ax.set_xticks(range(len(network_order)))
ax.set_yticks(range(len(network_order)))

ax.set_xticklabels(
    network_order,
    rotation=45,
    ha="right",
    fontsize=9
)

ax.set_yticklabels(
    network_order,
    fontsize=9
)

ax.set_title(
    "Active − Sham Difference in Network Transition",
    fontsize=12
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Difference in mean Δ Fisher-z connectivity")

plt.tight_layout()

plt.savefig(
    "Fig4A_Group_Difference_Heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig4A_Group_Difference_Heatmap.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# -------------------------------------------------------
# Figure 4B: Whole-brain transition magnitude
# Active vs Sham
# -------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.5, 5.5))

active_values = delta_group_df.loc[
    delta_group_df["Group"] == "active",
    "Mean_Absolute_DeltaFC"
].to_numpy()

sham_values = delta_group_df.loc[
    delta_group_df["Group"] == "sham",
    "Mean_Absolute_DeltaFC"
].to_numpy()

# Boxplot
ax.boxplot(
    [active_values, sham_values],
    tick_labels=["Active", "Sham"],
    widths=0.5
)

# Individual participants
rng = np.random.default_rng(42)

for x, values in zip([1, 2], [active_values, sham_values]):
    jitter = rng.normal(0, 0.035, size=len(values))
    ax.scatter(
        np.full(len(values), x) + jitter,
        values,
        s=40,
        zorder=3
    )

ax.set_ylabel("Mean absolute ΔFC")
ax.set_xlabel("Neurofeedback condition")

ax.set_title(
    "Whole-Brain Network Reorganization"
)

# Statistical result from existing analysis
ax.text(
    0.5,
    0.97,
    "Welch $t$ = 0.83, $p$ = 0.421",
    transform=ax.transAxes,
    ha="center",
    va="top"
)

plt.tight_layout()

plt.savefig(
    "Fig4B_WholeBrain_Group_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig4B_WholeBrain_Group_Comparison.svg",
    bbox_inches="tight"
)

plt.show()

print(f"Active: n={len(active_values)}, mean={active_values.mean():.4f}")
print(f"Sham:   n={len(sham_values)}, mean={sham_values.mean():.4f}")

In [ ]:
# =======================================================
# FINAL FIGURE 4
# Group-level brain-network reorganization
# =======================================================

import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    gridspec_kw={"width_ratios": [1.15, 0.85]}
)

# -------------------------------------------------------
# Panel A: Active - Sham network transition difference
# -------------------------------------------------------

ax = axes[0]

matrix = network_difference.to_numpy()
limit = np.nanmax(np.abs(matrix))

im = ax.imshow(
    matrix,
    vmin=-limit,
    vmax=limit
)

ax.set_xticks(range(len(network_order)))
ax.set_yticks(range(len(network_order)))

ax.set_xticklabels(
    network_order,
    rotation=45,
    ha="right",
    fontsize=8
)

ax.set_yticklabels(
    network_order,
    fontsize=8
)

ax.set_title(
    "(A) Active − Sham Network Transition",
    fontsize=11
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Difference in mean Δ Fisher-z connectivity",
    fontsize=9
)

# -------------------------------------------------------
# Panel B: whole-brain transition magnitude
# -------------------------------------------------------

ax = axes[1]

active_values = delta_group_df.loc[
    delta_group_df["Group"] == "active",
    "Mean_Absolute_DeltaFC"
].to_numpy()

sham_values = delta_group_df.loc[
    delta_group_df["Group"] == "sham",
    "Mean_Absolute_DeltaFC"
].to_numpy()

ax.boxplot(
    [active_values, sham_values],
    tick_labels=["Active", "Sham"],
    widths=0.5
)

rng = np.random.default_rng(42)

for x, values in zip(
    [1, 2],
    [active_values, sham_values]
):
    jitter = rng.normal(
        0,
        0.035,
        size=len(values)
    )

    ax.scatter(
        np.full(len(values), x) + jitter,
        values,
        s=35,
        zorder=3
    )

ax.set_ylabel(
    "Mean absolute ΔFC"
)

ax.set_xlabel(
    "Neurofeedback condition"
)

ax.set_title(
    "(B) Whole-Brain Reorganization",
    fontsize=11
)

ax.text(
    0.5,
    0.97,
    "Welch $t$ = 0.83, $p$ = 0.421",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=9
)

# -------------------------------------------------------
# Export
# -------------------------------------------------------

plt.tight_layout()

plt.savefig(
    "Fig4_Group_Level_Reorganization.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig4_Group_Level_Reorganization.svg",
    bbox_inches="tight"
)

plt.show()